<a href="https://colab.research.google.com/github/Vihhycherezass/TemperatureTimeSeriesForecast-/blob/main/Inteface%20(Streamlit).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 31.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from sklearn.preprocessing import MinMaxScaler

st.title('Предсказание температуры')


MODEL_PATH_OLD = '/content/drive/MyDrive/lstm_temperature_model.keras'
SCALER_PATH_OLD = '/content/drive/MyDrive/scaler.save'
WINDOW_PATH_OLD = '/content/drive/MyDrive/window_params.save'

MODEL_PATH_NEW = "new_model.keras"
SCALER_PATH_NEW = "new_scaler.save"
WINDOW_PATH_NEW = "new_window_params.save"

def create_sequences(data, window_size):
  X = []
  y = []

  for i in range(len(data) - window_size):
    X.append(data[i:i + window_size])
    y.append(data[i + window_size])

  return np.array(X), np.array(y)

def build_model(input_shape):
  model = tf.keras.Sequential([
      tf.keras.layers.LSTM(64, input_shape=input_shape),
      tf.keras.layers.Dense(32, activation='relu'),
      tf.keras.layers.Dense(1)
  ])

  model.compile(optimizer='adam', loss='mse', metrics=['mae'])
  return model

def prediction_next_temperature(model, scaler, window_data, target_index):
  window_scaled = scaler.transform(window_data)
  window_scaled = np.expand_dims(window_scaled, axis=0)
  pred_scaled = model.predict(window_scaled, verbose=0)

  temp_min = scaler.data_min_[target_index]
  temp_max = scaler.data_max_[target_index]
  pred_temp = pred_scaled[0][0] * (temp_max - temp_min)
  return pred_temp

st.header("Загрузка датасета для обучения")

train_file = st.file_uploader("Загрущите CSV для обучения модели", type='csv', key='train')

if train_file is not None:
  df_train = pd.read_csv(train_file)
  if "temperature" not in df_train.columns:
    st.error("В CSV должна быть колонка temperature")
    st.stop()

  if "time" in df_train.columns:
    df_train["time"] = pd.to_datetime(df_train["time"])
    df_train = df_train.sort_values("time")
    df_train = df_train.drop(columns=["time"])

  feature_columns = df_train.columns.tolist()
  target_index = feature_columns.index("temperature")
  window_size = 24

  scaler_new = MinMaxScaler()
  scaled_data = scaler_new.fit_transform(df_train[feature_columns])

  X_train, y_train = create_sequences(scaled_data, window_size)
  st.write(f"Форма обучающих данных {X_train.shape}")
  st.header("Обучение модели")

  epochs = st.slider("Эпохи", 1, 20, 5)

  if st.button("Обучить модель"):
    model_new = build_model((window_size, X_train.shape[2]))
    with st.spinner("Тренировка..."):
      model_new.fit(X_train, y_train, epochs=epochs, batch_size=64, validation_split=0.1)
    model_new.save(MODEL_PATH_NEW)
    joblib.dump(scaler_new, SCALER_PATH_NEW)
    window_params_new = {
        "window_size": window_size,
        "feature_columns": feature_columns,
        "target_index": target_index
    }
    joblib.dump(window_params_new, WINDOW_PATH_NEW)
    st.success("Новая модель обучена и сохранена!")

st.header("Загрузка старой модели")
model_old, scaler_old, window_old = None, None, None
try:
  model_old = tf.keras.models.load_model(MODEL_PATH_OLD)
  scaler_old = joblib.load(SCALER_PATH_OLD)
  window_old = joblib.load(WINDOW_PATH_OLD)
  st.success('Старая модель загружена')
except:
  st.warning("Старая модель или файлы scaler/window не найдены")

st.header("Предсказание температуры")
predict_file = st.file_uploader("Загрузите CSV для предсказания (минимум 24 строки)", type="csv", key="predict")
if predict_file is not None:
  df_pred = pd.read_csv(predict_file)

  model_choice = st.radio(
      "Выберите модель для предсказания",
      ("Старая модель", "Новая модель")
  )

  if model_choice == "Старая модель":
    if model_old is None:
      st.error("Старая модель не загуржена")
      st.stop()
    model_use = model_old
    scaler_use = scaler_old
    window_params_use = window_old
  else:
    try:
      model_use = tf.keras.models.load_model(MODEL_PATH_NEW)
      scaler_use = joblib.load(SCALER_PATH_NEW)
      window_params_use = joblib.load(WINDOW_PATH_NEW)
    except:
      st.error("Новая модель еще не обучена")
      st.stop()

  seq = df_pred[window_params_use['feature_columns']].tail(window_params_use['window_size']).values

  if len(seq) < window_params_use['window_size']:
    st.error(f"В файле должно быть хотя бы {window_params_use['window_size']} строк")
    st.stop()

  if st.button("Предсказать температуру"):
    result = prediction_next_temperature(model_use, scaler_use, seq, window_params_use['target_index'])
    st.success(f"Следующая температура: {result:.2f}°C")

Overwriting app.py


In [ ]:
!streamlit run app.py --server.address=localhost >/content/logs.txt & ssh -o "StrictHostKeyChecking no" -R 80:localhost:8501 serveo.net

Forwarding HTTP traffic from https://5f34849d0824ffac-34-169-105-139.serveousercontent.com
2026-02-05 03:44:59.050495: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-05 03:44:59.055318: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-05 03:44:59.069706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770263099.095263    1819 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770263099.104406    1819 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770263099.123485    1819 computation_